In [57]:
import pandas as pd
import numpy as np
#%pip install catboost

# For importing Evaluation metrics
#%pip install scikit-learn 


# Reading the files
train_transaction = pd.read_csv("../../Data/train_transaction.csv")
train_identity = pd.read_csv( "../../Data/train_identity.csv")

# Merging the two files 
train_merged = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)


In [ ]:
# Created the 1st [feature profile]

# Added new column to identifiy which transactions contain identity information 
train_merged["HasIdentity"] = (train_merged["TransactionID"].isin(train_identity["TransactionID"]).astype(int))
# Added a new column to display the date/time in a readable format ..
train_merged["TransactionDateTime"] = pd.to_datetime( train_merged["TransactionDT"],unit="s",origin="2017-11-30")

# Created the feature profile, then we will add more columns to (feature_profile)
feature_profile = pd.DataFrame({
    "Feature": train_merged.columns,
    "Data_Type": train_merged.dtypes.astype(str).values,
    "Missing_Count": train_merged.isna().sum().values,
    "Missing_Percent": (train_merged.isna().mean().values * 100 ).round(2),
    "Unique_Count": train_merged.nunique(dropna=True).values
})


feature_profile = feature_profile[~feature_profile["Feature"].isin(["TransactionID", "isFraud"])].reset_index(drop=True)

# feature_profile.head(10)

# MARAM
#transaction_columns = set(train_transaction.columns)
#identity_columns = set(train_identity.columns)

### Function to identify the source of a feature
def identify_source(feature_name):
    if feature_name in set(train_transaction.columns):
        return "Transaction"
    elif feature_name in set(train_identity.columns):
        return "Identity"
    else:
        return "Custom"
#################### End Function


# Function to classify the columns/feature

def identify_feature_group(feature_name):
    if feature_name in ["P_emaildomain", "R_emaildomain"]:
        return "Email"
    elif feature_name in ["DeviceType", "DeviceInfo"]:
        return "Device"
    elif feature_name.startswith("id_"):
        return "Identity_ID"
    elif feature_name.startswith("C") and feature_name[1:].isdigit():
        return "C_Features"
    elif feature_name.startswith("D") and feature_name[1:].isdigit():
        return "D_Features"
    elif feature_name.startswith("M") and feature_name[1:].isdigit():
        return "M_Features"
    elif feature_name.startswith("V") and feature_name[1:].isdigit():
        return "V_Features"
    elif feature_name  in ["HasIdentity", "TransactionDateTime"]:
        return "Custom"
    else:
        return feature_name 
    ############################### End Function


## Created two new column by applying the previous two functions 
feature_profile["Source"] = (feature_profile["Feature"].apply(identify_source))
feature_profile["Feature_Group"] = (feature_profile["Feature"].apply(identify_feature_group))


feature_profile = feature_profile[
    [
        # The column name
        "Feature",
        ## The two new coulmns created by applying the functiosns
        "Source",
        "Feature_Group",
        ############
        "Data_Type",
        "Missing_Count",
        "Missing_Percent",
        "Unique_Count"
    ]
]

group_summary = (
    feature_profile
    .groupby(["Source", "Feature_Group"])
    .agg(
        Feature_Count=("Feature", "count"),
        Average_Missing_Percent=("Missing_Percent", "mean"),
        Median_Missing_Percent=("Missing_Percent", "median"),
        Maximum_Missing_Percent=("Missing_Percent", "max")
    ).reset_index().round(2).sort_values(
        by="Feature_Count",
        ascending=False
    )
)

group_summary



# MARAM

# Calculated fraud rates for missing and available values of each feature
#target = train_merged["isFraud"]
missing_signal_results = []

for feature in feature_profile["Feature"]:
    #missing_mask = train_merged[feature].isna()
    #fraud_rate_missing = ( target[missing_mask].mean() * 100)
    #fraud_rate_available = ( target[~missing_mask].mean() * 100)
    fraud_rate_missing = (train_merged.loc[train_merged[feature].isna(), "isFraud"].mean() * 100 )
    fraud_rate_available = (train_merged.loc[train_merged[feature].notna(), "isFraud"].mean() * 100 )
    difference = fraud_rate_available - fraud_rate_missing

    missing_signal_results.append({
        "Feature": feature,
        "Fraud_Rate_Missing": fraud_rate_missing,
        "Fraud_Rate_Available": fraud_rate_available,
        "Fraud_Rate_Difference": difference
    })


missing_signal_df = pd.DataFrame(missing_signal_results)

feature_profile = feature_profile.merge(
    missing_signal_df,
    on="Feature",
    how="left"
)


rate_columns = [ "Fraud_Rate_Missing", "Fraud_Rate_Available","Fraud_Rate_Difference"]

feature_profile[rate_columns] = (feature_profile[rate_columns].round(3))
feature_profile["Available_Count"] = (len(train_merged)- feature_profile["Missing_Count"])


# Keep features with at least 1000 rows in both groups
minimum_group_size = 1000

stable_missing_signal = (
    feature_profile[
        (feature_profile["Missing_Count"] >= minimum_group_size)
        &
        (feature_profile["Available_Count"] >= minimum_group_size)
    ]
    .sort_values(by="Fraud_Rate_Difference",ascending=False)
)

# Calculateي the % of unique values among available values
feature_profile["Unique_Ratio"] = (
    feature_profile["Unique_Count"] /feature_profile["Available_Count"].replace(0, pd.NA)* 100
).round(2)

# Assigned a default quality flag
feature_profile["Quality_Flag"] = "Standard"

feature_profile.loc[ 
    feature_profile["Available_Count"] == 0,"Quality_Flag"
    ] = "All Missing"

feature_profile.loc[
    (feature_profile["Available_Count"] > 0)
    &
    (feature_profile["Unique_Count"] == 1),
    "Quality_Flag"
] = "Constant"

feature_profile.loc[
    (feature_profile["Unique_Count"] > 1)
    &
    (feature_profile["Unique_Ratio"] >= 95),
    "Quality_Flag"
] = "Near Unique"

feature_profile.loc[
    (feature_profile["Unique_Count"] >= 1000)
    &
    (feature_profile["Unique_Ratio"] < 95),
    "Quality_Flag"
] = "High Cardinality"

# Display features with a quality flag other than Standard
feature_profile[
    feature_profile["Quality_Flag"] != "Standard"
][
    [
        "Feature",
        "Source",
        "Feature_Group",
        "Data_Type",
        "Available_Count",
        "Unique_Count",
        "Unique_Ratio",
        "Quality_Flag"
    ]
].sort_values(
    by=["Quality_Flag", "Unique_Ratio"],
    ascending=[True, False]
)



    # Check time 

train_merged["TranHour"] = ( train_merged["TransactionDateTime"].dt.hour.astype("int8"))
train_merged["DayOfWeek"] = (train_merged["TransactionDateTime"].dt.dayofweek.astype("int8"))


# Fraud rate by time features - 
def summarize_fraud_by_group(data, group_column):
    summary = (
        data
        .groupby(group_column)["isFraud"]
        .agg(
            Transaction_Count="count",
            Fraud_Count="sum",
            Fraud_Rate="mean"
        )
        .reset_index()
    )
    summary["Fraud_Rate"] = (
        summary["Fraud_Rate"] * 100
    ).round(3)

    return summary
######### END FUNCTION

time_columns = [
    "TranHour",
    "DayOfWeek"
]

time_fraud_summaries = {}

for x in time_columns:

    time_fraud_summaries[x] = (
        summarize_fraud_by_group(
            train_merged,
            x
        )
    )

    print(f"\nFraud analysis by {x}:")

    display(
        time_fraud_summaries[x]
    )





In [78]:
## Train-Validation Split

#Checked whether TransactionDT is sorted in ascending order
train_merged["TransactionDT"].is_monotonic_increasing


split_index = int(len(train_merged) * 0.80)
#print(split_index)
#472,432

# Copy the the rows from 1 to 472,431 (80%)
train_data = train_merged.iloc[:split_index].copy()

# Copy the other 20% of data 
validation_data = train_merged.iloc[split_index:].copy()


print("All rows:",len(train_data) + len(validation_data) == len(train_merged))
print("# Training rows:",len(train_data))
print("# validation rows:", len(validation_data))

print("First training date:", train_data["TransactionDateTime"].min())
print("Last training date:", train_data["TransactionDateTime"].max())
print("First validation date:",validation_data["TransactionDateTime"].min())
print("Last validation date:",validation_data["TransactionDateTime"].max())

print("Training fraud rate:", train_data["isFraud"].mean() * 100)
print("Validation fraud rate:",validation_data["isFraud"].mean() * 100)



All rows: True
# Training rows: 472432
# validation rows: 118108
First training date: 2017-12-01 00:00:00
Last training date: 2018-04-20 02:54:02
First validation date: 2018-04-20 02:55:00
Last validation date: 2018-05-31 23:58:51
Training fraud rate: 3.5135215226741625
Validation fraud rate: 3.4409184813899145


In [60]:
# Exclude the ID, Fraud target, and original time columns from the baseline features
excluded_columns = [
    "TransactionID",
    "isFraud",
    "TransactionDT",
    "TransactionDateTime"
]

# Keep all remaining columns as baseline features
baseline_features = [
    column
    for column in train_merged.columns
    if column not in excluded_columns
]

#  Separate features (X) and target (y) for training
X_train = train_data.loc[:, baseline_features]
y_train = train_data.loc[:, "isFraud"]

X_validation = validation_data.loc[:, baseline_features]
y_validation = validation_data.loc[:, "isFraud"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

# --------------------------------------------
# Identified categorical features from the training data
categorical_features = (X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist())

# Identified numeric and boolean features from the training data
numeric_features = ( X_train.select_dtypes(include=["number", "bool"]).columns.tolist())

print("String/Cat features:", len(categorical_features))
print("Numeric/Bool features:", len(numeric_features))



print("string features:")
print(categorical_features[:15])

print("numeric features:")
print(numeric_features[:15])

X_train: (472432, 434)
y_train: (472432,)
X_validation: (118108, 434)
y_validation: (118108,)
String/Cat features: 31
Numeric/Bool features: 403
string features:
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12']
numeric features:
['TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6']


In [61]:
from catboost import CatBoostClassifier


X_train_model = X_train.copy(deep=False)
X_validation_model = X_validation.copy(deep=False)

for x in categorical_features:
    X_train_model[x] = (
        X_train_model[x].fillna("Missing").astype(str)
    )

    X_validation_model[x] = (
        X_validation_model[x].fillna("Missing").astype(str)
    )

print("Missing categorical values in train:",
    X_train_model[categorical_features].isna().sum().sum())

print("Missing categorical values in validation:",
    X_validation_model[categorical_features].isna().sum().sum())

Missing categorical values in train: 0
Missing categorical values in validation: 0


In [62]:
# Training the fraud model (V1)

fraud_model = CatBoostClassifier(
    iterations=200,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    task_type="CPU",
    allow_writing_files=False,
    verbose=25
)

fraud_model.fit(
    X_train_model,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_validation_model, y_validation),
    early_stopping_rounds=30,
    use_best_model=True
)

print("Best iteration:", fraud_model.get_best_iteration())

print(
    "Best validation AUC:",
    fraud_model.get_best_score()["validation"]["AUC"]
)

0:	test: 0.6388898	best: 0.6388898 (0)	total: 921ms	remaining: 3m 3s
25:	test: 0.8515384	best: 0.8515384 (25)	total: 23.8s	remaining: 2m 39s
50:	test: 0.8648742	best: 0.8648742 (50)	total: 47.2s	remaining: 2m 17s
75:	test: 0.8705335	best: 0.8705335 (75)	total: 1m 10s	remaining: 1m 54s
100:	test: 0.8757384	best: 0.8757384 (100)	total: 1m 33s	remaining: 1m 31s
125:	test: 0.8793795	best: 0.8793795 (125)	total: 1m 56s	remaining: 1m 8s
150:	test: 0.8812802	best: 0.8812802 (150)	total: 2m 19s	remaining: 45.3s
175:	test: 0.8818352	best: 0.8820233 (165)	total: 2m 43s	remaining: 22.2s
199:	test: 0.8832062	best: 0.8834239 (195)	total: 3m 8s	remaining: 0us

bestTest = 0.8834239388
bestIteration = 195

Shrink model to first 196 iterations.
Best iteration: 195
Best validation AUC: 0.8834239388316625


In [63]:
# Threshold Evaluation

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

fraud_probability = fraud_model.predict_proba(
    X_validation_model
)[:, 1]

print(fraud_model.predict_proba(
    X_validation_model
))

print(fraud_model.predict_proba(
    X_validation_model
)[:, 1])


# Best Candidate Threshold (0.2)
# It achieved the highest [F1-score] among the tested thresholds (between fraud detection recall and alert precision

thresholds = [0.50, 0.40, 0.30, 0.20, 0.10]
threshold_x = []


# These metrics use probabilities, so they do not change by threshold


# AUC
#1.0: A perfect model 
#0.5: Performance equal to random guessing
#Below 0.5: Performance is bad
roc_auc = roc_auc_score(y_validation,fraud_probability)
average_precision = average_precision_score(y_validation,fraud_probability)


for threshold in thresholds:



    # Converted fraud probability into 0 or 1
    fraud_prediction = (fraud_probability >= threshold ).astype(int)

    # Alarm Precision
    # When the model raises an alert, it is usually correct (thresholds = 0.5)
    # but, it misses many fraudulent transactions
    # Precision = TP/(TP+FP) ==> 1184/(1184 + 295)
    # Precision = 0.8005
    # Precision = TP / (TP + FP)
    precision = precision_score(y_validation,fraud_prediction,zero_division=0)

    # expected fraud rate 
    # recall = TP/(TP+FN) ==> 1184(1184+2880)
    recall = recall_score(y_validation,fraud_prediction,zero_division=0)


    # Balance between Precision and Recall
    # 2 * ((precision * recall) / (precision + recall) )
    f1 = f1_score(y_validation,fraud_prediction,zero_division=0)


    # Confusion Matrix values
    tn, fp, fn, tp = confusion_matrix(
    y_validation,
    fraud_prediction,
    labels=[0, 1]
    #Matrix [[113749,    295],
    #       [  2880,   1184]]
    ).ravel() # Ravel: [113749, 295, 2880, 1184] 

     #print("Thresdold:", Threshold_prob)
     #print("ROC-AUC:", round(roc_auc, 4)) 
     #print("Average Precision:", round(average_precision, 4))
     #print("Precision:", round(precision, 4))
     #print("Recall:", round(recall, 4))
     #print("F1-score:", round(f1, 4))
     #print("\nConfusion Matrix Counts")
     #print("True Negatives:", tn)
     # False Alarms
     #print("False Positives:", fp)
     #print("False Negatives:", fn)
     #print("True Positives:", tp)

    #print("\nActual fraud transactions:", y_validation.sum())
    #print("Predicted fraud transactions:", fraud_prediction.sum())

    # Add the current threshold results as one row
    threshold_x.append({
        "Threshold": threshold,
        "ROC_AUC": round(roc_auc, 2),
        "Average_Precision": round(average_precision, 2),
        "Precision": round(precision, 2),
        "Recall": round(recall, 2),
        "F1_Score": round(f1, 2),
        "True_Negative": int(tn),
        "False_Positive": int(fp),
        "False_Negative": int(fn),
        "True_Positive": int(tp),
        "Actual_Fraud": int(y_validation.sum()),
        "Predicted_Fraud": int(fraud_prediction.sum())
    })


# Convert all threshold results into one DataFrame
threshold_comparison = pd.DataFrame(threshold_x)

display(threshold_comparison)







[[0.96731916 0.03268084]
 [0.99487905 0.00512095]
 [0.71259546 0.28740454]
 ...
 [0.99496659 0.00503341]
 [0.95468415 0.04531585]
 [0.97779911 0.02220089]]
[0.03268084 0.00512095 0.28740454 ... 0.00503341 0.04531585 0.02220089]


,Threshold,ROC_AUC,Average_Precision,Precision,Recall,F1_Score,True_Negative,False_Positive,False_Negative,True_Positive,Actual_Fraud,Predicted_Fraud
0,0.5,0.88,0.49,0.79,0.29,0.42,113740,304,2886,1178,4064,1482
1,0.4,0.88,0.49,0.73,0.32,0.45,113573,471,2762,1302,4064,1773
2,0.3,0.88,0.49,0.66,0.36,0.47,113289,755,2593,1471,4064,2226
3,0.2,0.88,0.49,0.57,0.43,0.49,112690,1354,2297,1767,4064,3121
4,0.1,0.88,0.49,0.38,0.56,0.45,110314,3730,1787,2277,4064,6007


In [64]:
# Compared the features/columns with thier importance score(in model)

feature_importance = pd.DataFrame({
    "Feature": X_train_model.columns,
    "Importance": fraud_model.feature_importances_
})

#print(fraud_model.feature_importances_)

# Sort features/columns from most to least important 
feature_importance = (
    feature_importance.sort_values(by="Importance",ascending=False).reset_index(drop=True)
)

# Calculated the running total of importance scores
feature_importance["Cumulative_Importance"] = (feature_importance["Importance"].cumsum())

# Total should be ~100
print("Total importance:",feature_importance["Importance"].sum())

# The top 20 rows (features) account for 50% of the model's total feature importance
#display(feature_importance.head(20))

feature_importance_missingness = (
    feature_importance.head(20)
    .merge(
        feature_profile[
            ["Feature","Missing_Percent"]
        ],on="Feature",how="left"
    )
)

display(feature_importance_missingness)

Total importance: 99.99999999999999


,Feature,Importance,Cumulative_Importance,Missing_Percent
0,C13,5.653009,5.653009,0.00
1,C1,4.736233,10.389242,0.00
2,C14,4.701543,15.090785,0.00
3,card6,3.657929,18.748714,0.27
4,M4,3.357252,22.105966,47.66
5,TransactionAmt,2.871265,24.977231,0.00
6,M5,2.766133,27.743364,59.35
7,DeviceInfo,2.124688,29.868052,79.91
8,V317,2.002450,31.870503,0.00
9,D2,1.967165,33.837668,47.55


In [65]:
# Save the model + export the data 
fraud_model.save_model(
    "catboost_fraud_baseline_v1.cbm"
)

threshold_comparison.to_csv(
    "threshold_comparison_v1.csv",
    index=False
)

feature_importance.to_csv(
    "feature_importance_v1.csv",
    index=False
)

feature_importance_missingness.to_csv(
    "feature_importance_missingness_v1.csv",
    index=False
)

In [69]:
# Model 2 with 600 trees 

fraud_model_2 = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    task_type="CPU",
    allow_writing_files=False,
    verbose=50
)

fraud_model_2.fit(
    X_train_model,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_validation_model, y_validation),
    early_stopping_rounds=30,
    use_best_model=True
)

print(
    "Best iteration:",
    fraud_model_2.get_best_iteration()
)

print(
    "Best validation AUC:",
    fraud_model_2.get_best_score()["validation"]["AUC"]
)

0:	test: 0.6388898	best: 0.6388898 (0)	total: 956ms	remaining: 9m 32s
50:	test: 0.8648742	best: 0.8648742 (50)	total: 50.9s	remaining: 9m 8s
100:	test: 0.8757384	best: 0.8757384 (100)	total: 1m 40s	remaining: 8m 15s
150:	test: 0.8812802	best: 0.8812802 (150)	total: 2m 31s	remaining: 7m 30s
200:	test: 0.8832193	best: 0.8834239 (195)	total: 3m 59s	remaining: 7m 54s
250:	test: 0.8848211	best: 0.8853505 (244)	total: 5m 34s	remaining: 7m 45s
300:	test: 0.8861998	best: 0.8865826 (286)	total: 7m 8s	remaining: 7m 5s
350:	test: 0.8867843	best: 0.8873988 (325)	total: 8m 17s	remaining: 5m 53s
400:	test: 0.8883039	best: 0.8884808 (395)	total: 9m 7s	remaining: 4m 31s
450:	test: 0.8908731	best: 0.8908731 (450)	total: 9m 54s	remaining: 3m 16s
500:	test: 0.8921310	best: 0.8921581 (499)	total: 10m 40s	remaining: 2m 6s
550:	test: 0.8936925	best: 0.8936925 (550)	total: 11m 12s	remaining: 59.8s
599:	test: 0.8946749	best: 0.8947098 (579)	total: 11m 44s	remaining: 0us

bestTest = 0.894709825
bestIteration =

In [79]:
# Model 1 Vs Model 2 performance

fraud_probability_1 = fraud_model.predict_proba(
    X_validation_model)[:, 1]

fraud_probability_2 = fraud_model_2.predict_proba(
    X_validation_model)[:, 1]


model_comparisonx = pd.DataFrame({
    "Model": [ "Model 1 - Baseline","Model 2 - 600 Iterations" ],
    "ROC_AUC": [
        roc_auc_score( y_validation, fraud_probability_1),
        roc_auc_score(  y_validation,fraud_probability_2)
    ],
    "Average_Precision": [
        average_precision_score(y_validation,fraud_probability_1),
        average_precision_score( y_validation,fraud_probability_2)
    ]
})

display(model_comparisonx.round(4))

,Model,ROC_AUC,Average_Precision
0,Model 1 - Baseline,0.8834,0.4933
1,Model 2 - 600 Iterations,0.8947,0.5311


In [ ]:
#display(group_summary)
#display(feature_profile.sort_values("Missing_Count",ascending=False).reset_index(drop=True).head(50))
#display(feature_profile[feature_profile["Feature"].isin(["addr1","addr2"])])
#display(feature_profile[feature_profile["Source"] == "Identity"])

#### Feature/column Analysis Summary

# Data Quality
*    No column was completely null
*    No column contained only one value.
*    Missing values are not distributed equally across all columns

##### ##### ##### ###### 

# Missingness Analysis
The amount of missing data varies significantly across feature groups:

   V_Features:  339 columns started with V, thier average missing rate is about 43%
   Identity_ID: average missing rate of approximately 85%.
   D_Features: have an average missing rate of approximately 58%.
   M_Features: have an average missing rate of approximately 50%.
   C_Features: contain no missing values.
   Card columns have low % missingness
   Device, distance, email, and identity columns contain high levels of missing data

To avoid unreliable comparisons, missing-versus-available fraud rates were evaluated only when both groups contained at least 1000 tran

##### ##### ##### ###### 


# Important Missingness Patterns
* `D7` had a fraud rate of approximately 14.88% when available, compared with 2.70% when missing.
* `D12`, `D13`, and `D14` also showed substantially higher fraud rates when values were available.
* `id_03` and `id_04` had fraud rates of approximately 10.72% when available, compared with 2.59% when missing.
* `dist2` had a fraud rate of approximately 9.92% when available, compared with 3.06% when missing.
* `R_emaildomain` had a fraud rate of approximately 8.18% when available, compared with 2.08% when missing.
* # `addr1` and `addr2` showed the opposite pattern: missing values had a fraud rate of approximately [11.78%], compared with [2.46%] when available.
   
   These results indicate that missingness is not random and can contain predictive information.

##### ##### ##### ###### 

# Identity Availability
The high missingness of identity features is expected because identity information is available for only a subset of transactions

##### ##### ##### ###### 


# Time Feature Analysis
Theses two features were created `TranHour` &`DayOfWeek`
transaction hour showed the clearest variation in fraud behavior in specific hours(5-9AM), so I kept the `TranHour` as model feature

##### ##### ##### ###### 


# Feature Selection Decisions

* `TransactionID` was excluded because it is an identifier rather than a behavioral feature.
* `isFraud` was separated as the target variable
* CatBoost was selected because it can work with both numerical features & categorical features, missing numerical values
* Missing categorical values were converted to the explicit category "Missing"

Model validation and feature importance were used to judge whether a feature was useful instead of relying only on its missing percentage



## Modeling Summary

Two CatBoost models were trained using a chronological 80/20 train-validation split
*    The Training has 472,432 rows with [3.51] fraud rate
*    The validation has 118,108 rows with [3.44] fraud rate 

Created two models, the diffrence between them was the number of trees 
*    The baseline model has [200] trees :
     achieved a ROC-AUC of 0.8834 and Average Precision of 0.4933
*    The 2nd model has [600] trees :
     improved the ROC-AUC to [0.89] and Average Precision to 0.53

A probability threshold of [0.20] provided the best tested F1-score of [0.52]
    Model 2 achieved:
    *  Precision: 57.56%
    *  Recall: 47.51%
    *  True Negative: 112,620
    *  True Positives: 1,931 
    *  False Positives: 1,424 (false alaram)
    *  False Negatives: 2,133

The top 20 features represented [46.70%] of the total model importance.
*    The strongest features included C13, C1, C14, card2, TransactionAmt, M4, addr1, card6, M5, and DeviceInfo
*    Features wiz high missingness were still useful, showing that decisions to keep/remove features 
     shouldt be based on the % missingness alone